# Unified Forecasting with sktime

Every forecasting library in Python has its own API: statsmodels uses
`model.fit().forecast()`, Prophet uses `model.fit(df).predict(future)`,
pmdarima uses `model.fit(y).predict(n_periods)`, and scikit-learn uses
`model.fit(X, y).predict(X)`.  Comparing models fairly means writing
adapter code for each one.

**sktime** solves this problem.  It provides a single, scikit-learn-compatible
interface for time series forecasting (and classification, annotation, etc.)
that wraps all of these backends behind a consistent API:

```python
forecaster.fit(y_train)
y_pred = forecaster.predict(fh)
```

This notebook covers:
1. What is sktime and why use it?
2. Core concepts: `ForecastingHorizon`, `temporal_train_test_split`
3. Basic workflow with multiple forecasters
4. Comparing models with the same API
5. Pipelines: `TransformedTargetForecaster`
6. Ensembling: combining multiple forecasters
7. Performance metrics

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

---
## 1. What is sktime?

sktime is a unified framework for time series analysis in Python.  Key features:

| Feature | Description |
|---------|-------------|
| **Unified API** | All forecasters share `.fit()`, `.predict()`, `.update()` |
| **scikit-learn compatible** | Pipelines, parameter tuning, cross-validation |
| **Wraps existing libraries** | statsmodels, pmdarima, Prophet, etc. |
| **Composable** | Pipelines, ensembles, transformations |
| **Multiple tasks** | Forecasting, classification, regression, annotation |

The key insight: instead of learning each library's API, learn sktime's
API once and access everything through it.

---
## 2. Load Data

sktime expects a pandas Series with a `PeriodIndex` or `DatetimeIndex`
as the target variable `y`.

In [ ]:
# Load airline passengers
df = pd.read_csv(DATA_DIR / "airline_passengers.csv")
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
# Convert to a Series with PeriodIndex (sktime's preferred format)
y = pd.Series(
    df["Thousands of Passengers"].values,
    index=pd.PeriodIndex(pd.to_datetime(df["Month"]), freq="M"),
    name="Passengers",
)
print(f"Type: {type(y)}")
print(f"Index type: {type(y.index)}")
print(f"Shape: {y.shape}")
y.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
y.plot(ax=ax)
ax.set_title("Airline Passengers (1949-1960)")
ax.set_ylabel("Thousands of Passengers")
plt.tight_layout()
plt.show()

print("Upward trend with increasing seasonal amplitude.")

---
## 3. Core Concepts: Train/Test Split and Forecasting Horizon

sktime provides `temporal_train_test_split` to split time series data
while respecting temporal order.  The `ForecastingHorizon` object tells
a forecaster *how far ahead* to predict.

In [ ]:
from sktime.split import temporal_train_test_split
from sktime.forecasting.base import ForecastingHorizon

# Hold out last 36 months
y_train, y_test = temporal_train_test_split(y, test_size=36)

print(f"Train: {len(y_train)} months  ({y_train.index[0]} to {y_train.index[-1]})")
print(f"Test:  {len(y_test)} months  ({y_test.index[0]} to {y_test.index[-1]})")

In [ ]:
# Create a ForecastingHorizon from the test index
fh = ForecastingHorizon(y_test.index, is_relative=False)

print(f"Forecasting horizon: {len(fh)} steps")
print(f"First step: {fh[0]}")
print(f"Last step:  {fh[-1]}")
print()
print("Alternatively, for relative horizons: fh = ForecastingHorizon([1, 2, ..., 36])")

In [ ]:
# Visualize the split
fig, ax = plt.subplots(figsize=(12, 5))
y_train.plot(ax=ax, label="Train")
y_test.plot(ax=ax, label="Test", color="tab:orange")
ax.axvline(y_test.index[0].to_timestamp(), color="red", linestyle="--", alpha=0.7, label="Split point")
ax.set_title("Temporal Train/Test Split")
ax.set_ylabel("Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. NaiveForecaster

The simplest baseline.  `NaiveForecaster` supports:
- `strategy="last"` — repeat the last observed value
- `strategy="mean"` — predict the historical mean
- `strategy="drift"` — extrapolate using the slope from first to last observation
- `sp=12` (seasonal period) — use `strategy="last"` to get seasonal naive

In [ ]:
from sktime.forecasting.naive import NaiveForecaster

# Seasonal Naive: repeat the last year's pattern
naive_seasonal = NaiveForecaster(strategy="last", sp=12)
naive_seasonal.fit(y_train)
y_pred_naive = naive_seasonal.predict(fh)

print(f"Predictions shape: {y_pred_naive.shape}")
y_pred_naive.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
y_train[-48:].plot(ax=ax, label="Train (last 4 years)")
y_test.plot(ax=ax, label="Test (actual)", color="tab:orange")
y_pred_naive.plot(ax=ax, label="Seasonal Naive", color="tab:red", linestyle="--")
ax.set_title("Seasonal Naive Forecast")
ax.set_ylabel("Passengers")
ax.legend()
plt.tight_layout()
plt.show()

print("Seasonal naive captures the seasonal shape but misses the upward trend.")

---
## 5. Exponential Smoothing

In [ ]:
from sktime.forecasting.exp_smoothing import ExponentialSmoothing

es = ExponentialSmoothing(
    trend="add",
    seasonal="add",
    sp=12,
)
es.fit(y_train)
y_pred_es = es.predict(fh)

fig, ax = plt.subplots(figsize=(12, 5))
y_train[-48:].plot(ax=ax, label="Train")
y_test.plot(ax=ax, label="Test", color="tab:orange")
y_pred_es.plot(ax=ax, label="Exp. Smoothing", color="tab:green", linestyle="--")
ax.set_title("Exponential Smoothing Forecast")
ax.set_ylabel("Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. AutoARIMA

sktime wraps pmdarima's `auto_arima` behind the same interface.

In [ ]:
from sktime.forecasting.arima import AutoARIMA

auto_arima = AutoARIMA(sp=12, suppress_warnings=True)
auto_arima.fit(y_train)
y_pred_arima = auto_arima.predict(fh)

# Access the fitted order
print("Fitted model summary:")
print(auto_arima.summary())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
y_train[-48:].plot(ax=ax, label="Train")
y_test.plot(ax=ax, label="Test", color="tab:orange")
y_pred_arima.plot(ax=ax, label="AutoARIMA", color="tab:purple", linestyle="--")
ax.set_title("AutoARIMA Forecast")
ax.set_ylabel("Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. ThetaForecaster

The Theta method decomposes the series into two "theta lines" (one that
captures the long-term trend, one that captures short-term dynamics)
and combines them.  It won the M3 forecasting competition.

In [ ]:
from sktime.forecasting.theta import ThetaForecaster

theta = ThetaForecaster(sp=12)
theta.fit(y_train)
y_pred_theta = theta.predict(fh)

fig, ax = plt.subplots(figsize=(12, 5))
y_train[-48:].plot(ax=ax, label="Train")
y_test.plot(ax=ax, label="Test", color="tab:orange")
y_pred_theta.plot(ax=ax, label="Theta", color="tab:brown", linestyle="--")
ax.set_title("Theta Forecaster")
ax.set_ylabel("Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. Comparing All Models

Because every forecaster shares the same API, comparing them is trivial:
loop over a dictionary of forecasters, fit, predict, compute metrics.

In [ ]:
from sktime.performance_metrics.forecasting import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
)

# Collect all predictions
forecasters = {
    "Seasonal Naive": NaiveForecaster(strategy="last", sp=12),
    "Exp. Smoothing": ExponentialSmoothing(trend="add", seasonal="add", sp=12),
    "AutoARIMA": AutoARIMA(sp=12, suppress_warnings=True),
    "Theta": ThetaForecaster(sp=12),
}

results = []
predictions = {}

for name, forecaster in forecasters.items():
    forecaster.fit(y_train)
    y_pred = forecaster.predict(fh)
    predictions[name] = y_pred

    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred, square_root=True)
    mape = mean_absolute_percentage_error(y_test, y_pred)

    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "MAPE": mape})
    print(f"{name:20s}  MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.4f}")

results_df = pd.DataFrame(results).set_index("Model").sort_values("MAE")
results_df

In [ ]:
# Visual comparison of all forecasts
fig, ax = plt.subplots(figsize=(14, 6))

y_train[-36:].plot(ax=ax, label="Train", color="tab:blue")
y_test.plot(ax=ax, label="Test (actual)", color="black", linewidth=2)

colors = ["tab:red", "tab:green", "tab:purple", "tab:brown"]
for (name, pred), color in zip(predictions.items(), colors):
    pred.plot(ax=ax, label=name, linestyle="--", color=color)

ax.set_title("All Forecasters vs Actuals")
ax.set_ylabel("Passengers")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart of errors
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric in zip(axes, ["MAE", "RMSE", "MAPE"]):
    results_df[metric].plot.barh(ax=ax, color="tab:blue", edgecolor="black")
    ax.set_title(metric)
    ax.invert_yaxis()

plt.suptitle("Model Comparison — Airline Passengers", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Pipelines: TransformedTargetForecaster

Just like scikit-learn pipelines chain preprocessing with a model,
sktime pipelines chain *transformations* with a forecaster.

A common use case: **detrend** the series before forecasting with a
model that assumes stationarity, then add the trend back to the
predictions.

In [ ]:
from sktime.forecasting.compose import TransformedTargetForecaster
from sktime.transformations.series.detrend import Deseasonalizer, Detrender
from sktime.forecasting.trend import PolynomialTrendForecaster

# Pipeline: remove trend, remove seasonality, forecast residuals, invert
pipe = TransformedTargetForecaster(
    [
        ("detrend", Detrender(forecaster=PolynomialTrendForecaster(degree=1))),
        ("deseason", Deseasonalizer(model="additive", sp=12)),
        ("forecaster", NaiveForecaster(strategy="drift")),
    ]
)

print("Pipeline steps:")
for name, step in pipe.steps:
    print(f"  {name}: {step.__class__.__name__}")

In [ ]:
pipe.fit(y_train)
y_pred_pipe = pipe.predict(fh)

mae_pipe = mean_absolute_error(y_test, y_pred_pipe)
rmse_pipe = mean_squared_error(y_test, y_pred_pipe, square_root=True)
mape_pipe = mean_absolute_percentage_error(y_test, y_pred_pipe)

print(f"Pipeline (Detrend + Deseason + Drift):")
print(f"  MAE:  {mae_pipe:.2f}")
print(f"  RMSE: {rmse_pipe:.2f}")
print(f"  MAPE: {mape_pipe:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
y_train[-48:].plot(ax=ax, label="Train")
y_test.plot(ax=ax, label="Test", color="tab:orange")
y_pred_pipe.plot(ax=ax, label="Pipeline", color="tab:red", linestyle="--")
ax.set_title("Pipeline Forecast (Detrend + Deseason + Drift)")
ax.set_ylabel("Passengers")
ax.legend()
plt.tight_layout()
plt.show()

print("The pipeline enables a simple drift forecaster to handle trend and seasonality.")

---
## 10. Ensembling Forecasts

Forecast combination (ensembling) often outperforms individual models.
sktime provides `EnsembleForecaster` to combine multiple forecasters.

In [ ]:
from sktime.forecasting.compose import EnsembleForecaster

ensemble = EnsembleForecaster(
    forecasters=[
        ("es", ExponentialSmoothing(trend="add", seasonal="add", sp=12)),
        ("arima", AutoARIMA(sp=12, suppress_warnings=True)),
        ("theta", ThetaForecaster(sp=12)),
    ],
    aggfunc="mean",  # simple average
)

ensemble.fit(y_train)
y_pred_ens = ensemble.predict(fh)

mae_ens = mean_absolute_error(y_test, y_pred_ens)
rmse_ens = mean_squared_error(y_test, y_pred_ens, square_root=True)
mape_ens = mean_absolute_percentage_error(y_test, y_pred_ens)

print(f"Ensemble (ES + AutoARIMA + Theta):")
print(f"  MAE:  {mae_ens:.2f}")
print(f"  RMSE: {rmse_ens:.2f}")
print(f"  MAPE: {mape_ens:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
y_train[-36:].plot(ax=ax, label="Train", color="tab:blue")
y_test.plot(ax=ax, label="Test (actual)", color="black", linewidth=2)

# Individual models
for name, pred in predictions.items():
    if name != "Seasonal Naive":
        pred.plot(ax=ax, linestyle=":", alpha=0.5, label=name)

# Ensemble
y_pred_ens.plot(ax=ax, label="Ensemble", color="tab:red", linewidth=2.5, linestyle="--")

ax.set_title("Ensemble vs Individual Forecasts")
ax.set_ylabel("Passengers")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

print("Ensembling averages out individual model errors, often reducing RMSE.")

---
## 11. Performance Metrics in sktime

sktime provides its own metric functions that work directly with
pandas Series (preserving the index).  Key metrics:

| Function | Metric |
|----------|--------|
| `mean_absolute_error` | MAE |
| `mean_squared_error` | MSE (set `square_root=True` for RMSE) |
| `mean_absolute_percentage_error` | MAPE |
| `median_absolute_error` | MdAE |

In [ ]:
# Add pipeline and ensemble to the results table
all_results = results.copy()
all_results.append({"Model": "Pipeline (Detrend+Drift)", "MAE": mae_pipe, "RMSE": rmse_pipe, "MAPE": mape_pipe})
all_results.append({"Model": "Ensemble (ES+ARIMA+Theta)", "MAE": mae_ens, "RMSE": rmse_ens, "MAPE": mape_ens})

final_df = pd.DataFrame(all_results).set_index("Model").sort_values("MAE")
print("Final Model Comparison (sorted by MAE):")
final_df

In [ ]:
# Highlight the best model
best_model = final_df.index[0]
print(f"Best model by MAE: {best_model}")
print(f"  MAE:  {final_df.loc[best_model, 'MAE']:.2f}")
print(f"  RMSE: {final_df.loc[best_model, 'RMSE']:.2f}")
print(f"  MAPE: {final_df.loc[best_model, 'MAPE']:.4f}")

---
## 12. Prediction Intervals

sktime forecasters support prediction intervals through `predict_interval()`.

In [ ]:
# Get 95% prediction interval from Exponential Smoothing
es2 = ExponentialSmoothing(trend="add", seasonal="add", sp=12)
es2.fit(y_train)
y_pred_es2 = es2.predict(fh)
y_interval = es2.predict_interval(fh, coverage=0.95)

print("Prediction interval columns:")
print(y_interval.columns.tolist())
y_interval.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
y_train[-24:].plot(ax=ax, label="Train")
y_test.plot(ax=ax, label="Test (actual)", color="tab:orange", marker="o", markersize=3)
y_pred_es2.plot(ax=ax, label="Forecast", color="tab:red", linestyle="--")

# Fill prediction interval
lower = y_interval.iloc[:, 0]
upper = y_interval.iloc[:, 1]
ax.fill_between(
    y_interval.index.to_timestamp(),
    lower.values,
    upper.values,
    alpha=0.2, color="tab:red", label="95% PI",
)

ax.set_title("Exponential Smoothing with 95% Prediction Interval")
ax.set_ylabel("Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## Summary

- **sktime** provides a unified API for time series forecasting:
  `forecaster.fit(y_train)` then `forecaster.predict(fh)`

- **Core concepts**:
  - `temporal_train_test_split` — split while respecting temporal order
  - `ForecastingHorizon` — absolute or relative forecast steps

- **Forecasters** tested:
  - `NaiveForecaster` — baseline (last, mean, drift, seasonal)
  - `ExponentialSmoothing` — wraps statsmodels ETS
  - `AutoARIMA` — wraps pmdarima
  - `ThetaForecaster` — M3 competition winner

- **Pipelines** via `TransformedTargetForecaster` let you chain
  detrending/deseasonalizing with any forecaster

- **Ensembling** via `EnsembleForecaster` averages multiple forecasters
  and often reduces forecast error

- **Prediction intervals** via `predict_interval()` quantify uncertainty

In [ ]:
print("Next notebook: Nixtla's statsforecast — blazing fast statistical forecasting.")